In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Set random seed for reproducibility
np.random.seed(42)

# Define the quadratic function: f(x,y) = x^2 + 100*y^2
# This creates an elongated ellipse where CG has a clear advantage

tmp = 100

def f(x, y):
    return x**2 + tmp * y**2

def gradient(x, y):
    return np.array([2 * x, 2 * tmp * y])


# Gradient Descent implementation
def gradient_descent(start, learning_rate=0.01, max_iter=1000, tol=1e-8):
    """Vanilla gradient descent with fixed learning rate."""
    path = [start]
    x = np.array(start, dtype=float)
    
    for _ in range(max_iter):
        grad = gradient(x[0], x[1])

        Q = np.array([[2, 0], [0, 2 * tmp]])
        stepsize = grad @ grad / (grad @ (Q @ grad))
        x_new = x - stepsize * grad
        
        # x_new = x - learning_rate * grad
        path.append(x_new.copy())
        
        if np.linalg.norm(x_new - x) < tol:
            break
        x = x_new
    
    return np.array(path), len(path) - 1

# Conjugate Gradient implementation (Fletcher-Reeves)
def conjugate_gradient(start, max_iter=1000, tol=1e-8):
    """Conjugate Gradient method for quadratic optimization."""
    path = [start]
    x = np.array(start, dtype=float)
    grad = gradient(x[0], x[1])
    d = -grad
    
    for i in range(max_iter):
        # Line search: exact for quadratic functions
        # For f(x) = x^T A x, the optimal step is: alpha = - (grad^T d) / (d^T A d)
        # Here A = [[2, 0], [0, 200]], so d^T A d = 2*d[0]^2 + 200*d[1]^2
        alpha = - (grad[0] * d[0] + grad[1] * d[1]) / (2 * d[0]**2 + 2 * tmp * d[1]**2)
        
        x_new = x + alpha * d
        path.append(x_new.copy())
        
        grad_new = gradient(x_new[0], x_new[1])
        
        if np.linalg.norm(grad_new) < tol:
            break
        
        # Fletcher-Reeves update
        beta = (grad_new[0] * grad_new[0] + grad_new[1] * grad_new[1]) / \
               (grad[0] * grad[0] + grad[1] * grad[1])
        d = -grad_new + beta * d
        
        grad = grad_new
        x = x_new
    
    return np.array(path), len(path) - 1

# Create a grid for plotting
x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = f(X, Y)

# Choose a starting point far from the minimum
start_point = np.array([1.8, 0.4])

# Run both algorithms
gd_path, gd_iter = gradient_descent(start_point, learning_rate=0.01)
cg_path, cg_iter = conjugate_gradient(start_point)

# Create the figure
fig, ax = plt.subplots(figsize=(12, 8))

# Plot contour lines
contour = ax.contour(X, Y, Z, levels=20, cmap='viridis')
# plt.clabel(contour, inline=True, fontsize=8)

plot_gd_path = True
plot_cg_path = True

if plot_gd_path:
    # Plot the paths
    ax.plot(gd_path[:, 0], gd_path[:, 1], 'o-', 
             color='red', label=f'Gradient Descent ({gd_iter} iterations)', 
             markersize=4, linewidth=1.5, markerfacecolor='white', markeredgecolor='red')

    # Add arrows to show direction (simplified to avoid compatibility issues)
    ax.quiver(gd_path[:-1, 0], gd_path[:-1, 1], 
              gd_path[1:, 0] - gd_path[:-1, 0], 
              gd_path[1:, 1] - gd_path[:-1, 1], 
              color='red', alpha=0.3, width=0.002, scale_units='xy', scale=1)

if plot_cg_path:
    ax.plot(cg_path[:, 0], cg_path[:, 1], 's-', 
             color='blue', label=f'Conjugate Gradient ({cg_iter} iterations)', 
             markersize=5, linewidth=1.5, markerfacecolor='white', markeredgecolor='blue')
    
    ax.quiver(cg_path[:-1, 0], cg_path[:-1, 1], 
              cg_path[1:, 0] - cg_path[:-1, 0], 
              cg_path[1:, 1] - cg_path[:-1, 1], 
              color='blue', alpha=0.3, width=0.002, scale_units='xy', scale=1)


# Customize the plot
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Create a custom colormap for the text box
cmap = LinearSegmentedColormap.from_list('custom', ['#f0f0f0', '#e0e0e0'])

plt.tight_layout()
plt.show()

# Print summary
print(f"Gradient Descent: {gd_iter} iterations")
print(f"Conjugate Gradient: {cg_iter} iterations")
print(f"Speedup: {gd_iter/cg_iter:.1f}x")